In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/gongbaoxin/common-crawl-840b/glove.840B.300d.txt
/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip
/kaggle/input/competitions/word2vec-nlp-tutorial/sampleSubmission.csv
/kaggle/input/competitions/word2vec-nlp-tutorial/unlabeledTrainData.tsv.zip
/kaggle/input/competitions/word2vec-nlp-tutorial/labeledTrainData.tsv.zip


In [2]:
import logging
import os
import re
import sys
import numpy as np
from itertools import chain
from gensim.models import KeyedVectors
import gensim
import pandas as pd
import torch
from bs4 import BeautifulSoup
from sklearn.model_selection import train_test_split
import pickle

# =================== 超参（和你本地保持一致） ===================
embed_size = 300
max_len = 512

# =================== Kaggle路径【自行核对修改】 ===================
TRAIN_PATH = "/kaggle/input/competitions/word2vec-nlp-tutorial/labeledTrainData.tsv.zip"
TEST_PATH = "/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip"
GLOVE_PATH = "/kaggle/input/datasets/gongbaoxin/common-crawl-840b/glove.840B.300d.txt"

# =================== 文本清洗函数（原版不动） ===================
def review_to_wordlist(review, remove_stopwords=False):
    review_text = BeautifulSoup(review, "lxml").get_text()
    review_text = re.sub("[^a-zA-Z]", " ", review_text)
    words = review_text.lower().split()
    return words

def encode_samples(tokenized_samples, word_to_idx):
    features = []
    for sample in tokenized_samples:
        feature = []
        for token in sample:
            if token in word_to_idx:
                feature.append(word_to_idx[token])
            else:
                feature.append(0)
        features.append(feature)
    return features

def pad_samples(features, maxlen=max_len, PAD=0):
    padded_features = []
    for feature in features:
        if len(feature) >= maxlen:
            padded_feature = feature[:maxlen]
        else:
            padded_feature = feature.copy()
            while len(padded_feature) < maxlen:
                padded_feature.append(PAD)
        padded_features.append(padded_feature)
    return padded_features

# =================== 主流程 ===================
os.makedirs("/kaggle/working/pickle", exist_ok=True)

train = pd.read_csv(TRAIN_PATH, header=0, delimiter="\t", quoting=3)
test = pd.read_csv(TEST_PATH, header=0, delimiter="\t", quoting=3)

clean_train_reviews, train_labels = [], []
for i, review in enumerate(train["review"]):
    clean_train_reviews.append(review_to_wordlist(review))
    train_labels.append(train["sentiment"][i])

clean_test_reviews = []
for review in test["review"]:
    clean_test_reviews.append(review_to_wordlist(review))

vocab = set(chain(*clean_train_reviews)) | set(chain(*clean_test_reviews))
vocab_size = len(vocab)

train_reviews, val_reviews, train_labels, val_labels = train_test_split(
    clean_train_reviews, train_labels, test_size=0.2, random_state=0)

# ===================【重点修改】适配Common Crawl 840B（glove-gensim分割逻辑） ===================
wvmodel = KeyedVectors(embed_size)
word_dict = {}
with open(GLOVE_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        tokens = line.split()
        if len(tokens) <= embed_size:
            continue
        try:
            vec = np.array(tokens[-embed_size:], dtype=np.float32)
            word = " ".join(tokens[:-embed_size])
            word_dict[word] = vec
        except ValueError:
            continue
wvmodel.add_vectors(list(word_dict.keys()), list(word_dict.values()))
print(f"GloVe加载完成，载入词总数：{len(wvmodel)}")
# =========================================================================================

word_to_idx = {word: i + 1 for i, word in enumerate(vocab)}
word_to_idx['<unk>'] = 0
idx_to_word = {i + 1: word for i, word in enumerate(vocab)}
idx_to_word[0] = '<unk>'

train_features = torch.tensor(pad_samples(encode_samples(train_reviews, word_to_idx)))
val_features = torch.tensor(pad_samples(encode_samples(val_reviews, word_to_idx)))
test_features = torch.tensor(pad_samples(encode_samples(clean_test_reviews, word_to_idx)))

train_labels = torch.tensor(train_labels)
val_labels = torch.tensor(val_labels)

# 构建Embedding权重矩阵
weight = torch.zeros(vocab_size + 1, embed_size)
hit = 0
for word, idx in word_to_idx.items():
    if word in wvmodel:
        weight[idx, :] = torch.from_numpy(wvmodel.get_vector(word))
        hit += 1
print(f"词表匹配成功向量：{hit}/{len(word_to_idx)}")

pickle_file = "/kaggle/working/pickle/imdb_glove.pickle3"
pickle.dump(
    [train_features, train_labels, val_features, val_labels, test_features, weight, word_to_idx, idx_to_word, vocab],
    open(pickle_file, 'wb'))
print('pickle文件生成完成！')

GloVe加载完成，载入词总数：2195895


/tmp/ipykernel_58/1117333043.py:114: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  weight[idx, :] = torch.from_numpy(wvmodel.get_vector(word))


词表匹配成功向量：77554/101400
pickle文件生成完成！


In [3]:
import logging
import os
import sys
import pickle
import time

import pandas as pd
import torch
from torch import nn
from torch import optim
from tqdm import tqdm
from sklearn.metrics import accuracy_score

# ====================== 超参数 ======================
num_epochs = 10
embed_size = 300
num_hiddens = 120
num_layers = 2
bidirectional = True
batch_size = 64
labels = 2
lr = 0.08
use_gpu = torch.cuda.is_available()
device = torch.device('cuda:0' if use_gpu else 'cpu')


class SentimentNet(nn.Module):
    def __init__(self, embed_size, num_hiddens, num_layers, bidirectional, weight, labels, use_gpu, **kwargs):
        super(SentimentNet, self).__init__(**kwargs)
        self.num_hiddens = num_hiddens
        self.num_layers = num_layers
        self.use_gpu = use_gpu
        self.bidirectional = bidirectional
        self.embedding = nn.Embedding.from_pretrained(weight)
        self.embedding.weight.requires_grad = False
        self.encoder = nn.GRU(input_size=embed_size, hidden_size=self.num_hiddens,
                               num_layers=num_layers, bidirectional=self.bidirectional, dropout=0)
        # 修复维度：双向拼接正反隐状态仅 num_hiddens * 2
        if self.bidirectional:
            self.decoder = nn.Linear(num_hiddens * 2, labels)
        else:
            self.decoder = nn.Linear(num_hiddens, labels)

    def forward(self, inputs):
        embeddings = self.embedding(inputs)
        states, hidden = self.encoder(embeddings.permute([1, 0, 2]))
        if self.bidirectional:
            encoding = torch.cat([hidden[-2], hidden[-1]], dim=1)
        else:
            encoding = hidden[-1]
        outputs = self.decoder(encoding)
        return outputs


if __name__ == '__main__':
    program = os.path.basename(sys.argv[0])
    logger = logging.getLogger(program)

    logging.basicConfig(format='%(asctime)s: %(levelname)s: %(message)s')
    logging.root.setLevel(level=logging.INFO)
    logger.info(r"running %s" % ''.join(sys.argv))

    logging.info('loading data...')
    pickle_file = '/kaggle/working/pickle/imdb_glove.pickle3'
    [train_features, train_labels, val_features, val_labels, test_features, weight, word_to_idx, idx_to_word,
     vocab] = pickle.load(open(pickle_file, 'rb'))
    logging.info('data loaded!')

    net = SentimentNet(embed_size=embed_size, num_hiddens=num_hiddens, num_layers=num_layers,
                       bidirectional=bidirectional, weight=weight,
                       labels=labels, use_gpu=use_gpu)
    net.to(device)
    loss_function = nn.CrossEntropyLoss()
    optimizer = optim.SGD(net.parameters(), lr=lr)

    train_set = torch.utils.data.TensorDataset(train_features, train_labels)
    val_set = torch.utils.data.TensorDataset(val_features, val_labels)
    test_set = torch.utils.data.TensorDataset(test_features, )

    train_iter = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_iter = torch.utils.data.DataLoader(val_set, batch_size=batch_size, shuffle=False)
    test_iter = torch.utils.data.DataLoader(test_set, batch_size=batch_size, shuffle=False)

    for epoch in range(num_epochs):
        start = time.time()
        train_loss, val_losses = 0, 0
        train_acc, val_acc = 0, 0
        n, m = 0, 0
        net.train()
        with tqdm(total=len(train_iter), desc='Epoch %d' % epoch) as pbar:
            for feature, label in train_iter:
                n += 1
                net.zero_grad()
                feature = feature.to(device)
                label = label.to(device)
                score = net(feature)
                loss = loss_function(score, label)
                loss.backward()
                optimizer.step()
                train_acc += accuracy_score(torch.argmax(score.cpu().data, dim=1), label.cpu())
                train_loss += loss

                pbar.set_postfix({'epoch': '%d' % (epoch),
                                  'train loss': '%.4f' % (train_loss.data / n),
                                  'train acc': '%.2f' % (train_acc / n)
                                  })
                pbar.update(1)

            net.eval()
            with torch.no_grad():
                for val_feature, val_label in val_iter:
                    m += 1
                    val_feature = val_feature.to(device)
                    val_label = val_label.to(device)
                    val_score = net(val_feature)
                    val_loss = loss_function(val_score, val_label)
                    val_acc += accuracy_score(torch.argmax(val_score.cpu().data, dim=1), val_label.cpu())
                    val_losses += val_loss
            end = time.time()
            runtime = end - start
            pbar.set_postfix({'epoch': '%d' % (epoch),
                              'train loss': '%.4f' % (train_loss.data / n),
                              'train acc': '%.2f' % (train_acc / n),
                              'val loss': '%.4f' % (val_losses.data / m),
                              'val acc': '%.2f' % (val_acc / m),
                              'time': '%.2f' % (runtime)})

    test_pred = []
    net.eval()
    with torch.no_grad():
        with tqdm(total=len(test_iter), desc='Prediction') as pbar:
            for test_feature, in test_iter:
                test_feature = test_feature.to(device)
                test_score = net(test_feature)
                test_pred.extend(torch.argmax(test_score.cpu().data, dim=1).numpy().tolist())
                pbar.update(1)

    # 读取测试集id生成提交文件
    TEST_PATH = "/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip"
    test_df = pd.read_csv(TEST_PATH, header=0, delimiter="\t", quoting=3)
    result_output = pd.DataFrame(data={"id": test_df["id"], "sentiment": test_pred})

    save_path = "/kaggle/working/gru.csv"
    result_output.to_csv(save_path, index=False, quoting=3)
    logging.info(f'result saved to {save_path}!')


INFO:colab_kernel_launcher.py:running /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py-f/root/.local/share/jupyter/runtime/kernel-67929315-f3bc-46cf-adfe-f4fad2596062.json
INFO:root:loading data...
INFO:root:data loaded!
Prediction: 100%|██████████| 391/391 [00:06<00:00, 61.71it/s]
INFO:root:result saved to /kaggle/working/gru.csv!


In [4]:
import logging
import os
import sys
import pickle
import time
import random
import numpy as np

import pandas as pd
import torch
from torch import nn
from torch import optim
from tqdm import tqdm
from sklearn.metrics import accuracy_score

# ====================== 超参数 ======================
num_epochs = 10
embed_size = 300
num_hiddens = 120
num_layers = 2
bidirectional = True
batch_size = 64
labels = 2
lr = 1e-3                  # 更换Adam，降低初始学习率
max_grad_norm = 5.0        # 梯度裁剪，防止梯度爆炸
dropout_rate = 0.2         # GRU层间dropout
use_emb_finetune = False   # 是否微调预训练GloVe
patience = 3               # 早停耐心值

use_gpu = torch.cuda.is_available()
device = torch.device('cuda:0' if use_gpu else 'cpu')

# 固定随机种子，保证实验可复现
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


class SentimentNet(nn.Module):
    def __init__(self, embed_size, num_hiddens, num_layers, bidirectional, weight, labels, use_gpu, dropout, **kwargs):
        super(SentimentNet, self).__init__(**kwargs)
        self.num_hiddens = num_hiddens
        self.num_layers = num_layers
        self.use_gpu = use_gpu
        self.bidirectional = bidirectional

        self.embedding = nn.Embedding.from_pretrained(weight)
        self.embedding.weight.requires_grad = use_emb_finetune

        self.encoder = nn.GRU(input_size=embed_size, hidden_size=self.num_hiddens,
                               num_layers=num_layers, bidirectional=self.bidirectional,
                               dropout=dropout if num_layers > 1 else 0, batch_first=False)

        if self.bidirectional:
            self.decoder = nn.Linear(num_hiddens * 2, labels)
        else:
            self.decoder = nn.Linear(num_hiddens, labels)

    def forward(self, inputs):
        embeddings = self.embedding(inputs)
        states, hidden = self.encoder(embeddings.permute([1, 0, 2]))
        if self.bidirectional:
            encoding = torch.cat([hidden[-2], hidden[-1]], dim=1)
        else:
            encoding = hidden[-1]
        outputs = self.decoder(encoding)
        return outputs


if __name__ == '__main__':
    program = os.path.basename(sys.argv[0])
    logger = logging.getLogger(program)
    logging.basicConfig(format='%(asctime)s: %(levelname)s: %(message)s')
    logging.root.setLevel(level=logging.INFO)
    logger.info(r"running %s" % ''.join(sys.argv))

    logging.info('loading data...')
    pickle_file = '/kaggle/working/pickle/imdb_glove.pickle3'
    [train_features, train_labels, val_features, val_labels, test_features, weight, word_to_idx, idx_to_word,
     vocab] = pickle.load(open(pickle_file, 'rb'))
    logging.info('data loaded!')

    net = SentimentNet(embed_size=embed_size, num_hiddens=num_hiddens, num_layers=num_layers,
                       bidirectional=bidirectional, weight=weight,
                       labels=labels, use_gpu=use_gpu, dropout=dropout_rate)
    net.to(device)

    loss_function = nn.CrossEntropyLoss()
    # 替换SGD为Adam，收敛更快更稳定
    optimizer = optim.Adam(net.parameters(), lr=lr)
    # 学习率衰减：每轮衰减0.95
    scheduler = optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.95)

    train_set = torch.utils.data.TensorDataset(train_features, train_labels)
    val_set = torch.utils.data.TensorDataset(val_features, val_labels)
    test_set = torch.utils.data.TensorDataset(test_features, )

    train_iter = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_iter = torch.utils.data.DataLoader(val_set, batch_size=batch_size, shuffle=False)
    test_iter = torch.utils.data.DataLoader(test_set, batch_size=batch_size, shuffle=False)

    best_val_acc = 0.0
    early_stop_count = 0

    for epoch in range(num_epochs):
        start = time.time()
        train_loss_total = 0.0
        val_loss_total = 0.0
        train_acc_total = 0.0
        val_acc_total = 0.0
        train_batch_num = 0
        val_batch_num = 0

        net.train()
        with tqdm(total=len(train_iter), desc=f'Epoch {epoch} Train') as pbar:
            for feature, label in train_iter:
                train_batch_num += 1
                net.zero_grad()
                feature = feature.to(device)
                label = label.to(device)
                score = net(feature)
                loss = loss_function(score, label)
                loss.backward()

                # 梯度裁剪，解决循环网络梯度爆炸
                torch.nn.utils.clip_grad_norm_(net.parameters(), max_grad_norm)
                optimizer.step()

                pred = torch.argmax(score, dim=1)
                train_acc_total += accuracy_score(pred.cpu(), label.cpu())
                train_loss_total += loss.item()

                pbar.set_postfix({
                    'train_loss': f'{train_loss_total / train_batch_num:.4f}',
                    'train_acc': f'{train_acc_total / train_batch_num:.4f}'
                })
                pbar.update(1)

        net.eval()
        with torch.no_grad():
            for val_feature, val_label in val_iter:
                val_batch_num += 1
                val_feature = val_feature.to(device)
                val_label = val_label.to(device)
                val_score = net(val_feature)
                val_loss = loss_function(val_score, val_label)
                val_pred = torch.argmax(val_score, dim=1)
                val_acc_total += accuracy_score(val_pred.cpu(), val_label.cpu())
                val_loss_total += val_loss.item()

        # 计算本轮指标
        avg_train_loss = train_loss_total / train_batch_num
        avg_train_acc = train_acc_total / train_batch_num
        avg_val_loss = val_loss_total / val_batch_num
        avg_val_acc = val_acc_total / val_batch_num
        runtime = time.time() - start

        logger.info(f"Epoch {epoch}: "
                    f"TrainLoss:{avg_train_loss:.4f} TrainAcc:{avg_train_acc:.4f} | "
                    f"ValLoss:{avg_val_loss:.4f} ValAcc:{avg_val_acc:.4f} Time:{runtime:.2f}s")

        # 保存最优模型 + 早停逻辑
        if avg_val_acc > best_val_acc:
            best_val_acc = avg_val_acc
            early_stop_count = 0
            torch.save(net.state_dict(), "/kaggle/working/best_gru_model.pth")
            logging.info(f"Save best model, best val acc: {best_val_acc:.4f}")
        else:
            early_stop_count += 1
            if early_stop_count >= patience:
                logging.info(f"Early stop trigger, patience={patience}")
                break

        scheduler.step()

    # 加载验证集最优权重进行预测（关键！不用最后一轮模型）
    net.load_state_dict(torch.load("/kaggle/working/best_gru_model.pth"))
    net.eval()

    test_pred = []
    with torch.no_grad():
        with tqdm(total=len(test_iter), desc='Prediction') as pbar:
            for test_feature, in test_iter:
                test_feature = test_feature.to(device)
                test_score = net(test_feature)
                batch_pred = torch.argmax(test_score.cpu(), dim=1).numpy().tolist()
                test_pred.extend(batch_pred)
                pbar.update(1)

    # 读取测试集id生成提交文件
    TEST_PATH = "/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip"
    test_df = pd.read_csv(TEST_PATH, header=0, delimiter="\t", quoting=3)
    result_output = pd.DataFrame(data={"id": test_df["id"], "sentiment": test_pred})

    save_path = "/kaggle/working/gru.csv"
    result_output.to_csv(save_path, index=False, quoting=3)
    logging.info(f'result saved to {save_path}! Best Val Acc = {best_val_acc:.4f}')


INFO:colab_kernel_launcher.py:running /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py-f/root/.local/share/jupyter/runtime/kernel-67929315-f3bc-46cf-adfe-f4fad2596062.json
INFO:root:loading data...
INFO:root:data loaded!
Epoch 0 Train: 100%|██████████| 313/313 [00:15<00:00, 19.71it/s, train_loss=0.4833, train_acc=0.7492]
INFO:colab_kernel_launcher.py:Epoch 0: TrainLoss:0.4833 TrainAcc:0.7492 | ValLoss:0.2978 ValAcc:0.8770 Time:17.13s
INFO:root:Save best model, best val acc: 0.8770
Epoch 1 Train: 100%|██████████| 313/313 [00:15<00:00, 19.76it/s, train_loss=0.2819, train_acc=0.8841]
INFO:colab_kernel_launcher.py:Epoch 1: TrainLoss:0.2819 TrainAcc:0.8841 | ValLoss:0.2543 ValAcc:0.8975 Time:17.11s
INFO:root:Save best model, best val acc: 0.8975
Epoch 2 Train: 100%|██████████| 313/313 [00:16<00:00, 19.49it/s, train_loss=0.2507, train_acc=0.8969]
INFO:colab_kernel_launcher.py:Epoch 2: TrainLoss:0.2507 TrainAcc:0.8969 | ValLoss:0.2524 ValAcc:0.8989 Time:17.35s
INFO:root:Save b

In [9]:
import logging
import os
import sys
import pickle
import time
import random
import numpy as np

import pandas as pd
import torch
from torch import nn
from torch import optim
from tqdm import tqdm

# ====================== 超参数 ======================
num_epochs = 10
embed_size = 300
num_hiddens = 120
num_layers = 2
bidirectional = True
batch_size = 64
labels = 2
lr = 1e-3
max_grad_norm = 5.0
dropout_rate = 0.2
head_dropout = 0.3        # 输出头dropout
use_emb_finetune = False
patience = 3
seed = 42

# 路径配置集中管理
PICKLE_PATH = '/kaggle/working/pickle/imdb_glove.pickle3'
MODEL_SAVE_PATH = '/kaggle/working/best_gru_model.pth'
RESULT_CSV_PATH = '/kaggle/working/gru.csv'
TEST_DATA_PATH = "/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip"

use_gpu = torch.cuda.is_available()
device = torch.device('cuda:0' if use_gpu else 'cpu')

# 固定随机种子
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


class SentimentNet(nn.Module):
    def __init__(self, embed_size, num_hiddens, num_layers, bidirectional, weight, labels, dropout, head_dropout, **kwargs):
        super(SentimentNet, self).__init__(**kwargs)
        self.num_hiddens = num_hiddens
        self.num_layers = num_layers
        self.bidirectional = bidirectional

        self.embedding = nn.Embedding.from_pretrained(weight)
        self.embedding.weight.requires_grad = use_emb_finetune

        self.encoder = nn.GRU(input_size=embed_size, hidden_size=self.num_hiddens,
                               num_layers=num_layers, bidirectional=self.bidirectional,
                               dropout=dropout if num_layers > 1 else 0, batch_first=False)

        mul = 2 if bidirectional else 1
        self.feature_dim = num_hiddens * mul * 3

        self.dropout = nn.Dropout(head_dropout)
        self.decoder = nn.Linear(self.feature_dim, labels)

    def forward(self, inputs):
        embeddings = self.embedding(inputs)
        embeddings = embeddings.permute(1, 0, 2)

        states, hidden = self.encoder(embeddings)

        if self.bidirectional:
            last_hn = torch.cat([hidden[-2], hidden[-1]], dim=1)
        else:
            last_hn = hidden[-1]

        avg_pool = torch.mean(states, dim=0)
        max_pool, _ = torch.max(states, dim=0)

        feat = torch.cat([last_hn, avg_pool, max_pool], dim=-1)
        feat = self.dropout(feat)
        outputs = self.decoder(feat)
        return outputs


if __name__ == '__main__':
    program = os.path.basename(sys.argv[0])
    logger = logging.getLogger(program)
    logging.basicConfig(format='%(asctime)s: %(levelname)s: %(message)s')
    logging.root.setLevel(level=logging.INFO)
    logger.info(f"running {''.join(sys.argv)}")

    os.makedirs(os.path.dirname(MODEL_SAVE_PATH), exist_ok=True)

    logging.info('loading data...')
    with open(PICKLE_PATH, 'rb') as f:
        data = pickle.load(f)
    train_features, train_labels, val_features, val_labels, test_features, weight, word_to_idx, idx_to_word, vocab = data
    logging.info('data loaded!')

    net = SentimentNet(embed_size=embed_size,
                       num_hiddens=num_hiddens,
                       num_layers=num_layers,
                       bidirectional=bidirectional,
                       weight=weight,
                       labels=labels,
                       dropout=dropout_rate,
                       head_dropout=head_dropout)
    net.to(device)

    loss_function = nn.CrossEntropyLoss()
    optimizer = optim.Adam(net.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

    train_set = torch.utils.data.TensorDataset(train_features, train_labels)
    val_set = torch.utils.data.TensorDataset(val_features, val_labels)
    test_set = torch.utils.data.TensorDataset(test_features, )

    train_iter = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_iter = torch.utils.data.DataLoader(val_set, batch_size=batch_size, shuffle=False)
    test_iter = torch.utils.data.DataLoader(test_set, batch_size=batch_size, shuffle=False)

    best_val_acc = 0.0
    early_stop_count = 0

    for epoch in range(num_epochs):
        start = time.time()
        train_loss_total = 0.0
        val_loss_total = 0.0
        train_correct = 0
        val_correct = 0
        train_total = 0
        val_total = 0

        net.train()
        with tqdm(total=len(train_iter), desc=f'Epoch {epoch} Train') as pbar:
            for feature, label in train_iter:
                feature = feature.to(device)
                label = label.to(device)
                optimizer.zero_grad()

                score = net(feature)
                loss = loss_function(score, label)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(net.parameters(), max_grad_norm)
                optimizer.step()

                train_loss_total += loss.item()
                pred = torch.argmax(score, dim=1)
                train_correct += (pred == label).sum().item()
                train_total += label.size(0)

                avg_loss = train_loss_total / (pbar.n + 1)
                avg_acc = train_correct / train_total
                pbar.set_postfix({'loss': f'{avg_loss:.4f}', 'acc': f'{avg_acc:.4f}'})
                pbar.update(1)

        net.eval()
        with torch.no_grad():
            for val_feature, val_label in val_iter:
                val_feature = val_feature.to(device)
                val_label = val_label.to(device)
                val_score = net(val_feature)
                val_loss = loss_function(val_score, val_label)
                val_loss_total += val_loss.item()

                val_pred = torch.argmax(val_score, dim=1)
                val_correct += (val_pred == val_label).sum().item()
                val_total += val_label.size(0)

        avg_train_loss = train_loss_total / len(train_iter)
        avg_train_acc = train_correct / train_total
        avg_val_loss = val_loss_total / len(val_iter)
        avg_val_acc = val_correct / val_total
        runtime = time.time() - start

        logger.info(f"Epoch {epoch}: "
                    f"TrainLoss:{avg_train_loss:.4f} TrainAcc:{avg_train_acc:.4f} | "
                    f"ValLoss:{avg_val_loss:.4f} ValAcc:{avg_val_acc:.4f} Time:{runtime:.2f}s")

        if avg_val_acc > best_val_acc:
            best_val_acc = avg_val_acc
            early_stop_count = 0
            torch.save(net.state_dict(), MODEL_SAVE_PATH)
            logger.info(f"Save best model, best val acc: {best_val_acc:.4f}")
        else:
            early_stop_count += 1
            if early_stop_count >= patience:
                logger.info(f"Early stop triggered, patience={patience}")
                break

        scheduler.step()

    net.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))
    net.eval()

    test_pred = []
    with torch.no_grad():
        with tqdm(total=len(test_iter), desc='Prediction') as pbar:
            for test_feature, in test_iter:
                test_feature = test_feature.to(device)
                test_score = net(test_feature)
                batch_pred = torch.argmax(test_score, dim=1).cpu().numpy().tolist()
                test_pred.extend(batch_pred)
                pbar.update(1)

    test_df = pd.read_csv(TEST_DATA_PATH, header=0, delimiter="\t", quoting=3)
    result_output = pd.DataFrame({"id": test_df["id"], "sentiment": test_pred})
    result_output.to_csv(RESULT_CSV_PATH, index=False, quoting=3)
    logger.info(f'result saved to {RESULT_CSV_PATH}! Best Val Acc = {best_val_acc:.4f}')


INFO:colab_kernel_launcher.py:running /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py-f/root/.local/share/jupyter/runtime/kernel-67929315-f3bc-46cf-adfe-f4fad2596062.json
INFO:root:loading data...
INFO:root:data loaded!
Epoch 0 Train: 100%|██████████| 313/313 [00:16<00:00, 18.82it/s, loss=0.4056, acc=0.8031]
INFO:colab_kernel_launcher.py:Epoch 0: TrainLoss:0.4056 TrainAcc:0.8031 | ValLoss:0.3768 ValAcc:0.8410 Time:17.92s
INFO:colab_kernel_launcher.py:Save best model, best val acc: 0.8410
Epoch 1 Train: 100%|██████████| 313/313 [00:17<00:00, 18.14it/s, loss=0.2806, acc=0.8863]
INFO:colab_kernel_launcher.py:Epoch 1: TrainLoss:0.2806 TrainAcc:0.8863 | ValLoss:0.2596 ValAcc:0.8946 Time:18.57s
INFO:colab_kernel_launcher.py:Save best model, best val acc: 0.8946
Epoch 2 Train: 100%|██████████| 313/313 [00:18<00:00, 16.92it/s, loss=0.2489, acc=0.8999]
INFO:colab_kernel_launcher.py:Epoch 2: TrainLoss:0.2489 TrainAcc:0.8999 | ValLoss:0.2838 ValAcc:0.8804 Time:19.85s
Epoch 3 Trai

In [10]:
import logging
import os
import sys
import pickle
import time
import random
import numpy as np

import pandas as pd
import torch
from torch import nn
from torch import optim
from tqdm import tqdm

# ====================== 超参数【专为IMDB调优】 ======================
num_epochs = 12
embed_size = 300
num_hiddens = 120
num_layers = 2
bidirectional = True
batch_size = 64
labels = 2
lr = 1e-3
max_grad_norm = 5.0
dropout_rate = 0.1       # 降低dropout，防止欠拟合
head_dropout = 0.1
use_emb_finetune = True   # 开启词向量微调【核心涨点】
patience = 3
seed = 42

# 路径配置
PICKLE_PATH = '/kaggle/working/pickle/imdb_glove.pickle3'
MODEL_SAVE_PATH = '/kaggle/working/best_gru_model.pth'
RESULT_CSV_PATH = '/kaggle/working/gru.csv'
TEST_DATA_PATH = "/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip"

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# 固定随机种子
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True


class SentimentNet(nn.Module):
    def __init__(self, embed_size, num_hiddens, num_layers, bidirectional, weight, labels, dropout, head_dropout, **kwargs):
        super(SentimentNet, self).__init__(**kwargs)
        self.num_hiddens = num_hiddens
        self.num_layers = num_layers
        self.bidirectional = bidirectional

        self.embedding = nn.Embedding.from_pretrained(weight)
        self.embedding.weight.requires_grad = use_emb_finetune

        self.encoder = nn.GRU(input_size=embed_size, hidden_size=self.num_hiddens,
                               num_layers=num_layers, bidirectional=self.bidirectional,
                               dropout=dropout if num_layers > 1 else 0, batch_first=False)

        # 回归原版最优：仅双向最后隐状态 2*hidden
        self.dropout = nn.Dropout(head_dropout)
        self.decoder = nn.Linear(num_hiddens * 2, labels)

    def forward(self, inputs):
        embeddings = self.embedding(inputs)
        embeddings = embeddings.permute(1, 0, 2)

        states, hidden = self.encoder(embeddings)

        # 标准双向GRU分类特征（最稳、不掉点）
        encoding = torch.cat([hidden[-2], hidden[-1]], dim=1)
        encoding = self.dropout(encoding)
        outputs = self.decoder(encoding)
        return outputs


if __name__ == '__main__':
    program = os.path.basename(sys.argv[0])
    logger = logging.getLogger(program)
    logging.basicConfig(format='%(asctime)s: %(levelname)s: %(message)s')
    logging.root.setLevel(level=logging.INFO)

    os.makedirs(os.path.dirname(MODEL_SAVE_PATH), exist_ok=True)

    logging.info('loading data...')
    with open(PICKLE_PATH, 'rb') as f:
        train_features, train_labels, val_features, val_labels, test_features, weight, word_to_idx, idx_to_word, vocab = pickle.load(f)
    logging.info('data loaded!')

    net = SentimentNet(embed_size=embed_size,
                       num_hiddens=num_hiddens,
                       num_layers=num_layers,
                       bidirectional=bidirectional,
                       weight=weight,
                       labels=labels,
                       dropout=dropout_rate,
                       head_dropout=head_dropout)
    net.to(device)

    loss_function = nn.CrossEntropyLoss()
    optimizer = optim.Adam(net.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.92)

    train_set = torch.utils.data.TensorDataset(train_features, train_labels)
    val_set = torch.utils.data.TensorDataset(val_features, val_labels)
    test_set = torch.utils.data.TensorDataset(test_features, )

    train_iter = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_iter = torch.utils.data.DataLoader(val_set, batch_size=batch_size, shuffle=False)
    test_iter = torch.utils.data.DataLoader(test_set, batch_size=batch_size, shuffle=False)

    best_val_acc = 0.0
    early_stop_count = 0

    for epoch in range(num_epochs):
        start = time.time()
        train_loss_total = 0.0
        val_loss_total = 0.0
        train_correct = 0
        val_correct = 0
        train_total = 0
        val_total = 0

        net.train()
        with tqdm(total=len(train_iter), desc=f'Epoch {epoch} Train') as pbar:
            for feature, label in train_iter:
                feature = feature.to(device)
                label = label.to(device)
                optimizer.zero_grad()

                score = net(feature)
                loss = loss_function(score, label)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(net.parameters(), max_grad_norm)
                optimizer.step()

                train_loss_total += loss.item()
                pred = torch.argmax(score, dim=1)
                train_correct += (pred == label).sum().item()
                train_total += label.size(0)

                avg_loss = train_loss_total / (pbar.n + 1)
                avg_acc = train_correct / train_total
                pbar.set_postfix({'loss': f'{avg_loss:.4f}', 'acc': f'{avg_acc:.4f}'})
                pbar.update(1)

        net.eval()
        with torch.no_grad():
            for val_feature, val_label in val_iter:
                val_feature = val_feature.to(device)
                val_label = val_label.to(device)
                val_score = net(val_feature)
                val_loss = loss_function(val_score, val_label)
                val_loss_total += val_loss.item()

                val_pred = torch.argmax(val_score, dim=1)
                val_correct += (val_pred == val_label).sum().item()
                val_total += val_label.size(0)

        avg_train_loss = train_loss_total / len(train_iter)
        avg_train_acc = train_correct / train_total
        avg_val_loss = val_loss_total / len(val_iter)
        avg_val_acc = val_correct / val_total
        runtime = time.time() - start

        logger.info(f"Epoch {epoch}: TrainLoss:{avg_train_loss:.4f} TrainAcc:{avg_train_acc:.4f} | ValLoss:{avg_val_loss:.4f} ValAcc:{avg_val_acc:.4f} Time:{runtime:.2f}s")

        if avg_val_acc > best_val_acc:
            best_val_acc = avg_val_acc
            early_stop_count = 0
            torch.save(net.state_dict(), MODEL_SAVE_PATH)
            logger.info(f"New Best Model! Val Acc = {best_val_acc:.4f}")
        else:
            early_stop_count += 1
            if early_stop_count >= patience:
                logger.info(f"Early Stop! Best Acc:{best_val_acc:.4f}")
                break

        scheduler.step()

    # 加载最优模型预测
    net.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))
    net.eval()

    test_pred = []
    with torch.no_grad():
        for test_feature, in tqdm(test_iter, desc="Predicting"):
            test_feature = test_feature.to(device)
            test_score = net(test_feature)
            batch_pred = torch.argmax(test_score, dim=1).cpu().numpy().tolist()
            test_pred.extend(batch_pred)

    test_df = pd.read_csv(TEST_DATA_PATH, header=0, delimiter="\t", quoting=3)
    res = pd.DataFrame({"id": test_df["id"], "sentiment": test_pred})
    res.to_csv(RESULT_CSV_PATH, index=False)
    logger.info(f"Finished! Best Val Acc: {best_val_acc:.4f}, Result saved.")


INFO:root:loading data...
INFO:root:data loaded!
Epoch 0 Train: 100%|██████████| 313/313 [00:19<00:00, 15.80it/s, loss=0.4235, acc=0.7885]
INFO:colab_kernel_launcher.py:Epoch 0: TrainLoss:0.4235 TrainAcc:0.7885 | ValLoss:0.2587 ValAcc:0.8966 Time:21.12s
INFO:colab_kernel_launcher.py:New Best Model! Val Acc = 0.8966
Epoch 1 Train: 100%|██████████| 313/313 [00:20<00:00, 15.05it/s, loss=0.1752, acc=0.9357]
INFO:colab_kernel_launcher.py:Epoch 1: TrainLoss:0.1752 TrainAcc:0.9357 | ValLoss:0.2415 ValAcc:0.9076 Time:22.14s
INFO:colab_kernel_launcher.py:New Best Model! Val Acc = 0.9076
Epoch 2 Train: 100%|██████████| 313/313 [00:20<00:00, 15.07it/s, loss=0.0636, acc=0.9801]
INFO:colab_kernel_launcher.py:Epoch 2: TrainLoss:0.0636 TrainAcc:0.9801 | ValLoss:0.2975 ValAcc:0.8954 Time:22.08s
Epoch 3 Train: 100%|██████████| 313/313 [00:20<00:00, 15.45it/s, loss=0.0184, acc=0.9954]
INFO:colab_kernel_launcher.py:Epoch 3: TrainLoss:0.0184 TrainAcc:0.9954 | ValLoss:0.4331 ValAcc:0.8838 Time:21.55s
Epoch

In [11]:
import logging
import os
import sys
import pickle
import time
import random
import numpy as np

import pandas as pd
import torch
from torch import nn
from torch import optim
from tqdm import tqdm

# ====================== 超参数【IMDB调优】 ======================
num_epochs = 12
embed_size = 300
num_hiddens = 120
num_layers = 2
bidirectional = True
batch_size = 64
labels = 2
lr = 1e-3
max_grad_norm = 5.0
dropout_rate = 0.1
head_dropout = 0.1
use_emb_finetune = True
patience = 3
seed = 42

# 路径
PICKLE_PATH = '/kaggle/working/pickle/imdb_glove.pickle3'
MODEL_SAVE_PATH = '/kaggle/working/best_gru_model.pth'
RESULT_CSV_PATH = '/kaggle/working/gru.csv'
TEST_DATA_PATH = "/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip"

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# 固定随机种子
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True


class SentimentNet(nn.Module):
    def __init__(self, embed_size, num_hiddens, num_layers, bidirectional, weight, labels, dropout, head_dropout, **kwargs):
        super(SentimentNet, self).__init__(**kwargs)
        self.num_hiddens = num_hiddens
        self.num_layers = num_layers
        self.bidirectional = bidirectional

        self.embedding = nn.Embedding.from_pretrained(weight)
        self.embedding.weight.requires_grad = use_emb_finetune

        self.encoder = nn.GRU(input_size=embed_size, hidden_size=self.num_hiddens,
                               num_layers=num_layers, bidirectional=self.bidirectional,
                               dropout=dropout if num_layers > 1 else 0, batch_first=False)

        self.dropout = nn.Dropout(head_dropout)
        self.decoder = nn.Linear(num_hiddens * 2, labels)

    def forward(self, inputs):
        embeddings = self.embedding(inputs)
        embeddings = embeddings.permute(1, 0, 2)

        states, hidden = self.encoder(embeddings)

        encoding = torch.cat([hidden[-2], hidden[-1]], dim=1)
        encoding = self.dropout(encoding)
        outputs = self.decoder(encoding)
        return outputs


if __name__ == '__main__':
    program = os.path.basename(sys.argv[0])
    logger = logging.getLogger(program)
    logging.basicConfig(format='%(asctime)s: %(levelname)s: %(message)s')
    logging.root.setLevel(level=logging.INFO)

    os.makedirs(os.path.dirname(MODEL_SAVE_PATH), exist_ok=True)

    logging.info('loading data...')
    with open(PICKLE_PATH, 'rb') as f:
        train_features, train_labels, val_features, val_labels, test_features, weight, word_to_idx, idx_to_word, vocab = pickle.load(f)
    logging.info('data loaded!')

    net = SentimentNet(embed_size=embed_size,
                       num_hiddens=num_hiddens,
                       num_layers=num_layers,
                       bidirectional=bidirectional,
                       weight=weight,
                       labels=labels,
                       dropout=dropout_rate,
                       head_dropout=head_dropout)
    net.to(device)

    loss_function = nn.CrossEntropyLoss()
    optimizer = optim.Adam(net.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.92)

    train_set = torch.utils.data.TensorDataset(train_features, train_labels)
    val_set = torch.utils.data.TensorDataset(val_features, val_labels)
    test_set = torch.utils.data.TensorDataset(test_features, )

    train_iter = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_iter = torch.utils.data.DataLoader(val_set, batch_size=batch_size, shuffle=False)
    test_iter = torch.utils.data.DataLoader(test_set, batch_size=batch_size, shuffle=False)

    best_val_acc = 0.0
    early_stop_count = 0

    for epoch in range(num_epochs):
        start = time.time()
        train_loss_total = 0.0
        val_loss_total = 0.0
        train_correct = 0
        val_correct = 0
        train_total = 0
        val_total = 0

        net.train()
        with tqdm(total=len(train_iter), desc=f'Epoch {epoch} Train') as pbar:
            for feature, label in train_iter:
                feature = feature.to(device)
                label = label.to(device)
                optimizer.zero_grad()

                score = net(feature)
                loss = loss_function(score, label)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(net.parameters(), max_grad_norm)
                optimizer.step()

                train_loss_total += loss.item()
                pred = torch.argmax(score, dim=1)
                train_correct += (pred == label).sum().item()
                train_total += label.size(0)

                avg_loss = train_loss_total / (pbar.n + 1)
                avg_acc = train_correct / train_total
                pbar.set_postfix({'loss': f'{avg_loss:.4f}', 'acc': f'{avg_acc:.4f}'})
                pbar.update(1)

        net.eval()
        with torch.no_grad():
            for val_feature, val_label in val_iter:
                val_feature = val_feature.to(device)
                val_label = val_label.to(device)
                val_score = net(val_feature)
                val_loss = loss_function(val_score, val_label)
                val_loss_total += val_loss.item()

                val_pred = torch.argmax(val_score, dim=1)
                val_correct += (val_pred == val_label).sum().item()
                val_total += val_label.size(0)

        avg_train_loss = train_loss_total / len(train_iter)
        avg_train_acc = train_correct / train_total
        avg_val_loss = val_loss_total / len(val_iter)
        avg_val_acc = val_correct / val_total
        runtime = time.time() - start

        logger.info(f"Epoch {epoch}: TrainLoss:{avg_train_loss:.4f} TrainAcc:{avg_train_acc:.4f} | ValLoss:{avg_val_loss:.4f} ValAcc:{avg_val_acc:.4f} Time:{runtime:.2f}s")

        if avg_val_acc > best_val_acc:
            best_val_acc = avg_val_acc
            early_stop_count = 0
            torch.save(net.state_dict(), MODEL_SAVE_PATH)
            logger.info(f"New Best Model! Val Acc = {best_val_acc:.4f}")
        else:
            early_stop_count += 1
            if early_stop_count >= patience:
                logger.info(f"Early Stop! Best Acc:{best_val_acc:.4f}")
                break

        scheduler.step()

    # 加载最优模型预测 & 修复输出格式（移除quoting，符合Kaggle要求）
    net.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))
    net.eval()

    test_pred = []
    with torch.no_grad():
        for test_feature, in tqdm(test_iter, desc="Predicting"):
            test_feature = test_feature.to(device)
            test_score = net(test_feature)
            batch_pred = torch.argmax(test_score, dim=1).cpu().numpy().tolist()
            test_pred.extend(batch_pred)

    test_df = pd.read_csv(TEST_DATA_PATH, sep="\t")
    result_output = pd.DataFrame({
        "id": test_df["id"],
        "sentiment": test_pred
    })
    # 标准竞赛输出，无多余引号
    result_output.to_csv(RESULT_CSV_PATH, index=False, encoding="utf-8")
    logger.info(f"Finished! Best Val Acc: {best_val_acc:.4f}, Result saved.")


INFO:root:loading data...
INFO:root:data loaded!
Epoch 0 Train: 100%|██████████| 313/313 [00:19<00:00, 15.78it/s, loss=0.4235, acc=0.7885]
INFO:colab_kernel_launcher.py:Epoch 0: TrainLoss:0.4235 TrainAcc:0.7885 | ValLoss:0.2587 ValAcc:0.8966 Time:21.14s
INFO:colab_kernel_launcher.py:New Best Model! Val Acc = 0.8966
Epoch 1 Train: 100%|██████████| 313/313 [00:20<00:00, 15.08it/s, loss=0.1752, acc=0.9357]
INFO:colab_kernel_launcher.py:Epoch 1: TrainLoss:0.1752 TrainAcc:0.9357 | ValLoss:0.2415 ValAcc:0.9076 Time:22.08s
INFO:colab_kernel_launcher.py:New Best Model! Val Acc = 0.9076
Epoch 2 Train: 100%|██████████| 313/313 [00:20<00:00, 15.11it/s, loss=0.0636, acc=0.9801]
INFO:colab_kernel_launcher.py:Epoch 2: TrainLoss:0.0636 TrainAcc:0.9801 | ValLoss:0.2975 ValAcc:0.8954 Time:22.02s
Epoch 3 Train: 100%|██████████| 313/313 [00:20<00:00, 15.43it/s, loss=0.0184, acc=0.9954]
INFO:colab_kernel_launcher.py:Epoch 3: TrainLoss:0.0184 TrainAcc:0.9954 | ValLoss:0.4331 ValAcc:0.8838 Time:21.58s
Epoch

In [12]:
import logging
import os
import sys
import pickle
import time
import random
import numpy as np

import pandas as pd
import torch
from torch import nn
from torch import optim
from tqdm import tqdm

# ====================== 超参数【稳健基线，优先运行】 ======================
num_epochs = 15
embed_size = 300
num_hiddens = 120
num_layers = 2
bidirectional = True
batch_size = 64
labels = 2
lr = 1e-3
max_grad_norm = 5.0
dropout_rate = 0.2
head_dropout = 0.2
use_emb_finetune = False   # 冻结GloVe词向量，小数据集防过拟合
patience = 4
seed = 42
weight_decay = 1e-4        # 新增L2正则

# 路径
PICKLE_PATH = '/kaggle/working/pickle/imdb_glove.pickle3'
MODEL_SAVE_PATH = '/kaggle/working/best_gru_model.pth'
RESULT_CSV_PATH = '/kaggle/working/gru.csv'
TEST_DATA_PATH = "/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip"

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# 固定随机种子
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True


class SentimentNet(nn.Module):
    def __init__(self, embed_size, num_hiddens, num_layers, bidirectional, weight, labels, dropout, head_dropout, **kwargs):
        super(SentimentNet, self).__init__(**kwargs)
        self.num_hiddens = num_hiddens
        self.num_layers = num_layers
        self.bidirectional = bidirectional

        self.embedding = nn.Embedding.from_pretrained(weight)
        self.embedding.weight.requires_grad = use_emb_finetune

        self.encoder = nn.GRU(input_size=embed_size, hidden_size=self.num_hiddens,
                               num_layers=num_layers, bidirectional=self.bidirectional,
                               dropout=dropout if num_layers > 1 else 0, batch_first=False)

        self.dropout = nn.Dropout(head_dropout)
        self.decoder = nn.Linear(num_hiddens * 2, labels)

    def forward(self, inputs):
        embeddings = self.embedding(inputs)
        embeddings = embeddings.permute(1, 0, 2)

        states, hidden = self.encoder(embeddings)
        encoding = torch.cat([hidden[-2], hidden[-1]], dim=1)
        encoding = self.dropout(encoding)
        outputs = self.decoder(encoding)
        return outputs


if __name__ == '__main__':
    program = os.path.basename(sys.argv[0])
    logger = logging.getLogger(program)
    logging.basicConfig(format='%(asctime)s: %(levelname)s: %(message)s')
    logging.root.setLevel(level=logging.INFO)

    os.makedirs(os.path.dirname(MODEL_SAVE_PATH), exist_ok=True)

    logging.info('loading data...')
    with open(PICKLE_PATH, 'rb') as f:
        train_features, train_labels, val_features, val_labels, test_features, weight, word_to_idx, idx_to_word, vocab = pickle.load(f)
    logging.info('data loaded!')

    net = SentimentNet(embed_size=embed_size,
                       num_hiddens=num_hiddens,
                       num_layers=num_layers,
                       bidirectional=bidirectional,
                       weight=weight,
                       labels=labels,
                       dropout=dropout_rate,
                       head_dropout=head_dropout)
    net.to(device)

    loss_function = nn.CrossEntropyLoss()
    # Adam调整beta参数，增加weight_decay正则，移除学习率衰减
    optimizer = optim.Adam(net.parameters(), lr=lr, betas=(0.9, 0.98), weight_decay=weight_decay)

    train_set = torch.utils.data.TensorDataset(train_features, train_labels)
    val_set = torch.utils.data.TensorDataset(val_features, val_labels)
    test_set = torch.utils.data.TensorDataset(test_features, )

    train_iter = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_iter = torch.utils.data.DataLoader(val_set, batch_size=batch_size, shuffle=False)
    test_iter = torch.utils.data.DataLoader(test_set, batch_size=batch_size, shuffle=False)

    best_val_acc = 0.0
    early_stop_count = 0

    for epoch in range(num_epochs):
        start = time.time()
        train_loss_total = 0.0
        val_loss_total = 0.0
        train_correct = 0
        val_correct = 0
        train_total = 0
        val_total = 0

        net.train()
        with tqdm(total=len(train_iter), desc=f'Epoch {epoch} Train') as pbar:
            for feature, label in train_iter:
                feature = feature.to(device)
                label = label.to(device)
                optimizer.zero_grad()

                score = net(feature)
                loss = loss_function(score, label)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(net.parameters(), max_grad_norm)
                optimizer.step()

                train_loss_total += loss.item()
                pred = torch.argmax(score, dim=1)
                train_correct += (pred == label).sum().item()
                train_total += label.size(0)

                avg_loss = train_loss_total / (pbar.n + 1)
                avg_acc = train_correct / train_total
                pbar.set_postfix({'loss': f'{avg_loss:.4f}', 'acc': f'{avg_acc:.4f}'})
                pbar.update(1)

        net.eval()
        with torch.no_grad():
            for val_feature, val_label in val_iter:
                val_feature = val_feature.to(device)
                val_label = val_label.to(device)
                val_score = net(val_feature)
                val_loss = loss_function(val_score, val_label)
                val_loss_total += val_loss.item()

                val_pred = torch.argmax(val_score, dim=1)
                val_correct += (val_pred == val_label).sum().item()
                val_total += val_label.size(0)

        avg_train_loss = train_loss_total / len(train_iter)
        avg_train_acc = train_correct / train_total
        avg_val_loss = val_loss_total / len(val_iter)
        avg_val_acc = val_correct / val_total
        runtime = time.time() - start

        logger.info(f"Epoch {epoch}: TrainLoss:{avg_train_loss:.4f} TrainAcc:{avg_train_acc:.4f} | ValLoss:{avg_val_loss:.4f} ValAcc:{avg_val_acc:.4f} Time:{runtime:.2f}s")

        if avg_val_acc > best_val_acc:
            best_val_acc = avg_val_acc
            early_stop_count = 0
            torch.save(net.state_dict(), MODEL_SAVE_PATH)
            logger.info(f"New Best Model! Val Acc = {best_val_acc:.4f}")
        else:
            early_stop_count += 1
            if early_stop_count >= patience:
                logger.info(f"Early Stop! Best Acc:{best_val_acc:.4f}")
                break

    # 预测输出，严格Kaggle标准格式
    net.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))
    net.eval()

    test_pred = []
    with torch.no_grad():
        for test_feature, in tqdm(test_iter, desc="Predicting"):
            test_feature = test_feature.to(device)
            test_score = net(test_feature)
            batch_pred = torch.argmax(test_score, dim=1).cpu().numpy().tolist()
            test_pred.extend(batch_pred)

    test_df = pd.read_csv(TEST_DATA_PATH, sep="\t")
    result_output = pd.DataFrame({
        "id": test_df["id"],
        "sentiment": test_pred
    })
    result_output.to_csv(RESULT_CSV_PATH, index=False, encoding="utf-8")
    logger.info(f"Finished! Best Val Acc: {best_val_acc:.4f}, Result saved.")


INFO:root:loading data...
INFO:root:data loaded!
Epoch 0 Train: 100%|██████████| 313/313 [00:16<00:00, 18.76it/s, loss=0.5383, acc=0.7264]
INFO:colab_kernel_launcher.py:Epoch 0: TrainLoss:0.5383 TrainAcc:0.7264 | ValLoss:0.3651 ValAcc:0.8606 Time:17.98s
INFO:colab_kernel_launcher.py:New Best Model! Val Acc = 0.8606
Epoch 1 Train: 100%|██████████| 313/313 [00:17<00:00, 17.46it/s, loss=0.3303, acc=0.8673]
INFO:colab_kernel_launcher.py:Epoch 1: TrainLoss:0.3303 TrainAcc:0.8673 | ValLoss:0.2909 ValAcc:0.8876 Time:19.26s
INFO:colab_kernel_launcher.py:New Best Model! Val Acc = 0.8876
Epoch 2 Train: 100%|██████████| 313/313 [00:18<00:00, 16.91it/s, loss=0.2855, acc=0.8844]
INFO:colab_kernel_launcher.py:Epoch 2: TrainLoss:0.2855 TrainAcc:0.8844 | ValLoss:0.2805 ValAcc:0.8870 Time:19.83s
Epoch 3 Train: 100%|██████████| 313/313 [00:17<00:00, 17.83it/s, loss=0.2564, acc=0.8992]
INFO:colab_kernel_launcher.py:Epoch 3: TrainLoss:0.2564 TrainAcc:0.8992 | ValLoss:0.2737 ValAcc:0.8894 Time:18.87s
INFO:

In [14]:
import logging
import os
import sys
import pickle
import time
import random
import numpy as np

import pandas as pd
import torch
from torch import nn
from torch import optim
from tqdm import tqdm

# ====================== 超参数 ======================
num_epochs = 18
embed_size = 300
num_hiddens = 120
num_layers = 2
bidirectional = True
batch_size = 64
labels = 2
lr = 1e-3
max_grad_norm = 5.0
dropout_rate = 0.2
head_dropout = 0.2
use_emb_finetune = False
patience = 5
seed = 42
weight_decay = 1e-4

# 路径
PICKLE_PATH = '/kaggle/working/pickle/imdb_glove.pickle3'
MODEL_SAVE_PATH = '/kaggle/working/best_gru_model.pth'
RESULT_CSV_PATH = '/kaggle/working/gru.csv'
TEST_DATA_PATH = "/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip"

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# 固定随机种子
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True


class SentimentNet(nn.Module):
    def __init__(self, embed_size, num_hiddens, num_layers, bidirectional, weight, labels, dropout, head_dropout, **kwargs):
        super().__init__(**kwargs)
        self.num_hiddens = num_hiddens
        self.bidirectional = bidirectional

        self.embedding = nn.Embedding.from_pretrained(weight)
        self.embedding.weight.requires_grad = use_emb_finetune

        self.gru = nn.GRU(input_size=embed_size, hidden_size=num_hiddens,
                          num_layers=num_layers, bidirectional=bidirectional,
                          dropout=dropout if num_layers > 1 else 0, batch_first=False)

        out_dim = num_hiddens * 2
        # 池化拼接：平均 + 最大
        self.dropout = nn.Dropout(head_dropout)
        self.fc1 = nn.Linear(out_dim * 2, out_dim)
        self.fc_out = nn.Linear(out_dim, labels)

    def forward(self, inputs):
        # inputs: [batch, seq_len]
        emb = self.embedding(inputs)  # [batch, seq_len, 300]
        emb = emb.permute(1, 0, 2)   # [seq_len, batch, embed]

        outputs, _ = self.gru(emb)   # outputs: [seq_len, batch, 2*hidden]
        outputs = outputs.permute(1, 0, 2)  # [batch, seq_len, hidden*2]

        avg_pool = torch.mean(outputs, dim=1)
        max_pool, _ = torch.max(outputs, dim=1)

        concat = torch.cat([avg_pool, max_pool], dim=1)
        feat = self.dropout(concat)
        feat = torch.relu(self.fc1(feat))
        feat = self.dropout(feat)
        logits = self.fc_out(feat)
        return logits


if __name__ == '__main__':
    program = os.path.basename(sys.argv[0])
    logger = logging.getLogger(program)
    logging.basicConfig(format='%(asctime)s: %(levelname)s: %(message)s')
    logging.root.setLevel(level=logging.INFO)

    os.makedirs(os.path.dirname(MODEL_SAVE_PATH), exist_ok=True)

    logging.info('loading data...')
    with open(PICKLE_PATH, 'rb') as f:
        train_features, train_labels, val_features, val_labels, test_features, weight, word_to_idx, idx_to_word, vocab = pickle.load(f)
    logging.info('data loaded!')

    net = SentimentNet(embed_size=embed_size,
                       num_hiddens=num_hiddens,
                       num_layers=num_layers,
                       bidirectional=bidirectional,
                       weight=weight,
                       labels=labels,
                       dropout=dropout_rate,
                       head_dropout=head_dropout)
    net.to(device)

    loss_function = nn.CrossEntropyLoss()
    optimizer = optim.Adam(net.parameters(), lr=lr, betas=(0.9, 0.98), weight_decay=weight_decay)
    # 移除verbose参数，兼容低版本PyTorch
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.7, patience=2)

    train_set = torch.utils.data.TensorDataset(train_features, train_labels)
    val_set = torch.utils.data.TensorDataset(val_features, val_labels)
    test_set = torch.utils.data.TensorDataset(test_features, )

    train_iter = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_iter = torch.utils.data.DataLoader(val_set, batch_size=batch_size, shuffle=False)
    test_iter = torch.utils.data.DataLoader(test_set, batch_size=batch_size, shuffle=False)

    best_val_acc = 0.0
    early_stop_count = 0

    for epoch in range(num_epochs):
        start = time.time()
        train_loss_total = 0.0
        val_loss_total = 0.0
        train_correct = 0
        val_correct = 0
        train_total = 0
        val_total = 0

        net.train()
        with tqdm(total=len(train_iter), desc=f'Epoch {epoch} Train') as pbar:
            for feature, label in train_iter:
                feature = feature.to(device)
                label = label.to(device)
                optimizer.zero_grad()

                score = net(feature)
                loss = loss_function(score, label)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(net.parameters(), max_grad_norm)
                optimizer.step()

                train_loss_total += loss.item()
                pred = torch.argmax(score, dim=1)
                train_correct += (pred == label).sum().item()
                train_total += label.size(0)

                avg_loss = train_loss_total / (pbar.n + 1)
                avg_acc = train_correct / train_total
                pbar.set_postfix({'loss': f'{avg_loss:.4f}', 'acc': f'{avg_acc:.4f}'})
                pbar.update(1)

        net.eval()
        with torch.no_grad():
            for val_feature, val_label in val_iter:
                val_feature = val_feature.to(device)
                val_label = val_label.to(device)
                val_score = net(val_feature)
                val_loss = loss_function(val_score, val_label)
                val_loss_total += val_loss.item()

                val_pred = torch.argmax(val_score, dim=1)
                val_correct += (val_pred == val_label).sum().item()
                val_total += val_label.size(0)

        avg_train_loss = train_loss_total / len(train_iter)
        avg_train_acc = train_correct / train_total
        avg_val_loss = val_loss_total / len(val_iter)
        avg_val_acc = val_correct / val_total
        runtime = time.time() - start

        logger.info(f"Epoch {epoch}: TrainLoss:{avg_train_loss:.4f} TrainAcc:{avg_train_acc:.4f} | ValLoss:{avg_val_loss:.4f} ValAcc:{avg_val_acc:.4f} Time:{runtime:.2f}s")

        scheduler.step(avg_val_acc)

        if avg_val_acc > best_val_acc:
            best_val_acc = avg_val_acc
            early_stop_count = 0
            torch.save(net.state_dict(), MODEL_SAVE_PATH)
            logger.info(f"✅ New Best Model! Val Acc = {best_val_acc:.4f}")
        else:
            early_stop_count += 1
            if early_stop_count >= patience:
                logger.info(f"🛑 Early Stop! Best Acc:{best_val_acc:.4f}")
                break

    # 预测输出，严格Kaggle标准格式
    net.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))
    net.eval()

    test_pred = []
    with torch.no_grad():
        for test_feature, in tqdm(test_iter, desc="Predicting"):
            test_feature = test_feature.to(device)
            test_score = net(test_feature)
            batch_pred = torch.argmax(test_score, dim=1).cpu().numpy().tolist()
            test_pred.extend(batch_pred)

    test_df = pd.read_csv(TEST_DATA_PATH, sep="\t")
    result_output = pd.DataFrame({
        "id": test_df["id"],
        "sentiment": test_pred
    })
    result_output.to_csv(RESULT_CSV_PATH, index=False, encoding="utf-8")
    logger.info(f"Finished! Best Val Acc: {best_val_acc:.4f}, Result saved.")


INFO:root:loading data...
INFO:root:data loaded!
Epoch 0 Train: 100%|██████████| 313/313 [00:17<00:00, 17.84it/s, loss=0.4133, acc=0.7947]
INFO:colab_kernel_launcher.py:Epoch 0: TrainLoss:0.4133 TrainAcc:0.7947 | ValLoss:0.2791 ValAcc:0.8844 Time:18.87s
INFO:colab_kernel_launcher.py:✅ New Best Model! Val Acc = 0.8844
Epoch 1 Train: 100%|██████████| 313/313 [00:18<00:00, 16.54it/s, loss=0.2893, acc=0.8788]
INFO:colab_kernel_launcher.py:Epoch 1: TrainLoss:0.2893 TrainAcc:0.8788 | ValLoss:0.2903 ValAcc:0.8836 Time:20.29s
Epoch 2 Train: 100%|██████████| 313/313 [00:19<00:00, 16.47it/s, loss=0.2622, acc=0.8931]
INFO:colab_kernel_launcher.py:Epoch 2: TrainLoss:0.2622 TrainAcc:0.8931 | ValLoss:0.2678 ValAcc:0.8934 Time:20.35s
INFO:colab_kernel_launcher.py:✅ New Best Model! Val Acc = 0.8934
Epoch 3 Train: 100%|██████████| 313/313 [00:18<00:00, 16.68it/s, loss=0.2402, acc=0.9045]
INFO:colab_kernel_launcher.py:Epoch 3: TrainLoss:0.2402 TrainAcc:0.9045 | ValLoss:0.2500 ValAcc:0.8982 Time:20.09s
I

In [15]:
import logging
import os
import sys
import pickle
import time
import random
import numpy as np

import pandas as pd
import torch
from torch import nn
from torch import optim
from tqdm import tqdm

# ====================== 超参数 ======================
num_epochs = 20
embed_size = 300
num_hiddens = 128
num_layers = 2
bidirectional = True
batch_size = 64
labels = 2
lr = 1e-3
max_grad_norm = 5.0
dropout_rate = 0.2
head_dropout = 0.25
use_emb_finetune = False
patience = 5
seed = 42
weight_decay = 1e-4

# 路径
PICKLE_PATH = '/kaggle/working/pickle/imdb_glove.pickle3'
MODEL_SAVE_PATH = '/kaggle/working/best_gru_model.pth'
RESULT_CSV_PATH = '/kaggle/working/gru.csv'
TEST_DATA_PATH = "/kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip"

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

# 固定随机种子
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True


class SentimentNet(nn.Module):
    def __init__(self, embed_size, num_hiddens, num_layers, bidirectional, weight, labels, dropout, head_dropout, **kwargs):
        super().__init__(**kwargs)
        self.num_hiddens = num_hiddens
        self.bidirectional = bidirectional

        self.embedding = nn.Embedding.from_pretrained(weight)
        self.embedding.weight.requires_grad = use_emb_finetune

        self.gru = nn.GRU(input_size=embed_size, hidden_size=num_hiddens,
                          num_layers=num_layers, bidirectional=bidirectional,
                          dropout=dropout if num_layers > 1 else 0, batch_first=True)

        out_dim = num_hiddens * 2
        # 三路特征融合：平均池化 + 最大池化 + 最后时刻隐状态
        self.dropout = nn.Dropout(head_dropout)
        self.fc1 = nn.Linear(out_dim * 3, out_dim)
        self.fc_out = nn.Linear(out_dim, labels)

    def forward(self, inputs):
        # inputs: [batch, seq_len]
        emb = self.embedding(inputs)  # [batch, seq_len, 300]
        # 生成padding掩码（非0为有效字符）
        mask = (inputs != 0).unsqueeze(-1).float()

        outputs, hidden = self.gru(emb)   # outputs: [batch, seq_len, 2*hidden]
        outputs = outputs * mask

        # 带掩码平均池化，规避padding干扰
        sum_feat = torch.sum(outputs, dim=1)
        len_valid = torch.clamp(torch.sum(mask, dim=1), min=1e-8)
        avg_pool = sum_feat / len_valid

        max_pool, _ = torch.max(outputs, dim=1)
        # 最后一层双向隐状态拼接
        last_hidden = torch.cat([hidden[-2], hidden[-1]], dim=-1)

        concat = torch.cat([avg_pool, max_pool, last_hidden], dim=1)
        feat = self.dropout(concat)
        feat = torch.relu(self.fc1(feat))
        feat = self.dropout(feat)
        logits = self.fc_out(feat)
        return logits


if __name__ == '__main__':
    program = os.path.basename(sys.argv[0])
    logger = logging.getLogger(program)
    logging.basicConfig(format='%(asctime)s: %(levelname)s: %(message)s')
    logging.root.setLevel(level=logging.INFO)

    os.makedirs(os.path.dirname(MODEL_SAVE_PATH), exist_ok=True)

    logging.info('loading data...')
    with open(PICKLE_PATH, 'rb') as f:
        train_features, train_labels, val_features, val_labels, test_features, weight, word_to_idx, idx_to_word, vocab = pickle.load(f)
    logging.info('data loaded!')

    net = SentimentNet(embed_size=embed_size,
                       num_hiddens=num_hiddens,
                       num_layers=num_layers,
                       bidirectional=bidirectional,
                       weight=weight,
                       labels=labels,
                       dropout=dropout_rate,
                       head_dropout=head_dropout)
    net.to(device)

    loss_function = nn.CrossEntropyLoss()
    # 标准Adam，去除容易震荡的beta参数
    optimizer = optim.Adam(net.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.7, patience=2)

    train_set = torch.utils.data.TensorDataset(train_features, train_labels)
    val_set = torch.utils.data.TensorDataset(val_features, val_labels)
    test_set = torch.utils.data.TensorDataset(test_features, )

    train_iter = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_iter = torch.utils.data.DataLoader(val_set, batch_size=batch_size, shuffle=False)
    test_iter = torch.utils.data.DataLoader(test_set, batch_size=batch_size, shuffle=False)

    best_val_acc = 0.0
    early_stop_count = 0

    for epoch in range(num_epochs):
        start = time.time()
        train_loss_total = 0.0
        val_loss_total = 0.0
        train_correct = 0
        val_correct = 0
        train_total = 0
        val_total = 0

        net.train()
        with tqdm(total=len(train_iter), desc=f'Epoch {epoch} Train') as pbar:
            for feature, label in train_iter:
                feature = feature.to(device)
                label = label.to(device)
                optimizer.zero_grad()

                score = net(feature)
                loss = loss_function(score, label)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(net.parameters(), max_grad_norm)
                optimizer.step()

                train_loss_total += loss.item()
                pred = torch.argmax(score, dim=1)
                train_correct += (pred == label).sum().item()
                train_total += label.size(0)

                avg_loss = train_loss_total / (pbar.n + 1)
                avg_acc = train_correct / train_total
                pbar.set_postfix({'loss': f'{avg_loss:.4f}', 'acc': f'{avg_acc:.4f}'})
                pbar.update(1)

        net.eval()
        with torch.no_grad():
            for val_feature, val_label in val_iter:
                val_feature = val_feature.to(device)
                val_label = val_label.to(device)
                val_score = net(val_feature)
                val_loss = loss_function(val_score, val_label)
                val_loss_total += val_loss.item()

                val_pred = torch.argmax(val_score, dim=1)
                val_correct += (val_pred == val_label).sum().item()
                val_total += val_label.size(0)

        avg_train_loss = train_loss_total / len(train_iter)
        avg_train_acc = train_correct / train_total
        avg_val_loss = val_loss_total / len(val_iter)
        avg_val_acc = val_correct / val_total
        runtime = time.time() - start

        logger.info(f"Epoch {epoch}: TrainLoss:{avg_train_loss:.4f} TrainAcc:{avg_train_acc:.4f} | ValLoss:{avg_val_loss:.4f} ValAcc:{avg_val_acc:.4f} Time:{runtime:.2f}s")

        scheduler.step(avg_val_acc)

        if avg_val_acc > best_val_acc:
            best_val_acc = avg_val_acc
            early_stop_count = 0
            torch.save(net.state_dict(), MODEL_SAVE_PATH)
            logger.info(f"✅ New Best Model! Val Acc = {best_val_acc:.4f}")
        else:
            early_stop_count += 1
            if early_stop_count >= patience:
                logger.info(f"🛑 Early Stop! Best Acc:{best_val_acc:.4f}")
                break

    # 预测输出
    net.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))
    net.eval()

    test_pred = []
    with torch.no_grad():
        for test_feature, in tqdm(test_iter, desc="Predicting"):
            test_feature = test_feature.to(device)
            test_score = net(test_feature)
            batch_pred = torch.argmax(test_score, dim=1).cpu().numpy().tolist()
            test_pred.extend(batch_pred)

    test_df = pd.read_csv(TEST_DATA_PATH, sep="\t")
    result_output = pd.DataFrame({
        "id": test_df["id"],
        "sentiment": test_pred
    })
    result_output.to_csv(RESULT_CSV_PATH, index=False, encoding="utf-8")
    logger.info(f"Finished! Best Val Acc: {best_val_acc:.4f}, Result saved.")


INFO:root:loading data...
INFO:root:data loaded!
Epoch 0 Train: 100%|██████████| 313/313 [00:18<00:00, 16.65it/s, loss=0.3833, acc=0.8175]
INFO:colab_kernel_launcher.py:Epoch 0: TrainLoss:0.3833 TrainAcc:0.8175 | ValLoss:0.3724 ValAcc:0.8368 Time:20.19s
INFO:colab_kernel_launcher.py:✅ New Best Model! Val Acc = 0.8368
Epoch 1 Train: 100%|██████████| 313/313 [00:18<00:00, 16.84it/s, loss=0.2840, acc=0.8841]
INFO:colab_kernel_launcher.py:Epoch 1: TrainLoss:0.2840 TrainAcc:0.8841 | ValLoss:0.2980 ValAcc:0.8812 Time:19.97s
INFO:colab_kernel_launcher.py:✅ New Best Model! Val Acc = 0.8812
Epoch 2 Train: 100%|██████████| 313/313 [00:18<00:00, 16.58it/s, loss=0.2552, acc=0.8975]
INFO:colab_kernel_launcher.py:Epoch 2: TrainLoss:0.2552 TrainAcc:0.8975 | ValLoss:0.2452 ValAcc:0.8986 Time:20.29s
INFO:colab_kernel_launcher.py:✅ New Best Model! Val Acc = 0.8986
Epoch 3 Train: 100%|██████████| 313/313 [00:19<00:00, 16.46it/s, loss=0.2397, acc=0.9042]
INFO:colab_kernel_launcher.py:Epoch 3: TrainLoss:0.